# Caso 1 · Cauce: clasificación automática de mensajes de clientes

**Equipo:** _completar nombres_

Este notebook sigue las consignas del enunciado (pasos 1 a 20). Cada paso tiene su celda; agreguen todas las que necesiten.
Las funciones de `src/` se completan en los archivos `.py`, no en el notebook. Antes de entregar: *Reiniciar y ejecutar todo*.

In [ ]:
%load_ext autoreload
%autoreload 2
# autoreload vuelve a cargar los módulos de src/ cada vez que los modifican, sin reiniciar el notebook.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEMILLA = 42

from src.limpieza import limpiar_texto
from src.naive_bayes import NaiveBayesMultinomial, terminos_mas_indicativos
from src.derivacion import REVISION, derivar, evaluar_derivacion

### Estado de las pruebas automáticas
Ejecuten esta celda cada vez que terminen una función de `src/`.

In [ ]:
!python -m pytest -q --tb=no -rf

## Parte A · Exploración y línea base

**Paso 1.** Carguen `datos/mensajes_historicos.csv` e informen filas, nulos, mensajes vacíos, duplicados (mismo `id_cliente` y mismo `texto`), distribución de `equipo_final`, distribución por canal y cantidad de palabras por clase.

In [ ]:
# Carga del dataset histórico
df = pd.read_csv("datos/mensajes_historicos.csv")

# 1. Total de filas y columnas
print(f"Total de filas: {df.shape[0]:,}")
print(f"Total de columnas: {df.shape[1]}")
print(f"Columnas disponibles: {list(df.columns)}")

# 2. Valores nulos por columna
print("\n--- Valores Nulos por Columna ---")
print(df.isnull().sum())

# 3. Mensajes vacíos o de solo espacios en blanco
mensajes_vacios = (df["texto"].astype(str).str.strip() == "").sum()
print(f"\nMensajes vacíos o de solo espacios en blanco: {mensajes_vacios}")

# 4. Duplicados (mismo id_cliente y mismo texto)
duplicados = df.duplicated(subset=["id_cliente", "texto"]).sum()
print(f"Mensajes duplicados (mismo id_cliente y mismo texto): {duplicados}")

# 5. Distribución de equipo_final (clase objetivo)
print("\n--- Distribución de equipo_final ---")
dist_equipo = pd.DataFrame({
    "Cantidad": df["equipo_final"].value_counts(),
    "Porcentaje (%)": (df["equipo_final"].value_counts(normalize=True) * 100).round(2)
})
display(dist_equipo)

# 6. Distribución por canal de contacto
print("\n--- Distribución por canal ---")
dist_canal = pd.DataFrame({
    "Cantidad": df["canal"].value_counts(),
    "Porcentaje (%)": (df["canal"].value_counts(normalize=True) * 100).round(2)
})
display(dist_canal)

# 7. Cantidad de palabras por clase (equipo_final)
df["cant_palabras"] = df["texto"].astype(str).str.split().str.len()
stats_palabras = df.groupby("equipo_final")["cant_palabras"].agg(
    Media="mean",
    Mediana="median",
    Desvio="std",
    Min="min",
    Max="max"
).round(2)
print("\n--- Cantidad de palabras por clase ---")
display(stats_palabras)


**Paso 2.** Línea base de la mesa de entrada (`equipo_derivado` contra `equipo_final`): exactitud, F1 macro, matriz de confusión normalizada por fila y porcentaje de fraude derivado a otro equipo.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Métricas globales de la mesa de entrada
acc_base = accuracy_score(df["equipo_final"], df["equipo_derivado"])
f1_base = f1_score(df["equipo_final"], df["equipo_derivado"], average="macro")

print("=== LÍNEA BASE: MESA DE ENTRADA HUMANA ===")
print(f"Exactitud (Accuracy): {acc_base:.4f} ({acc_base * 100:.2f}%)")
print(f"F1 Macro:             {f1_base:.4f} ({f1_base * 100:.2f}%)\n")

# Matriz de confusión normalizada por fila (proporción sobre el total real de cada clase)
clases = sorted(df["equipo_final"].unique())
cm_norm = confusion_matrix(df["equipo_final"], df["equipo_derivado"], labels=clases, normalize="true")
df_cm = pd.DataFrame(
    cm_norm,
    index=[f"Real: {c}" for c in clases],
    columns=[f"Derivado: {c}" for c in clases]
).round(4)

print("Matriz de confusión normalizada por fila (tasa de acierto y desvíos):")
display(df_cm)

# Análisis de fraude derivado a otro equipo
total_fraude = (df["equipo_final"] == "fraude").sum()
fraude_mal_derivado = ((df["equipo_final"] == "fraude") & (df["equipo_derivado"] != "fraude")).sum()
pct_fraude_fuga = (fraude_mal_derivado / total_fraude) * 100

print(f"Total casos de Fraude:                {total_fraude}")
print(f"Fraude derivado a OTRO equipo:        {fraude_mal_derivado}")
print(f"Porcentaje de Fraude derivado a otro: {pct_fraude_fuga:.2f}%")


**Paso 3.** Eliminen los duplicados conservando el primero y expliquen por qué se hace antes de dividir.

> **Justificación metodológica:** La deduplicación por `(id_cliente, texto)` debe realizarse **estrictamente antes de la partición de datos (train/test split)** para prevenir el fenómeno de **fuga de datos (data leakage)**. Si no se removieran previamente, un mensaje idéntico emitido por un mismo cliente podría quedar asignado simultáneamente al conjunto de entrenamiento y al de prueba. Esto provocaría una evaluación sesgada sobre datos ya memorizados por el modelo, sobreestimando artificialmente las métricas de exactitud y F1 macro sin reflejar su desempeño real ante nuevos mensajes.

In [ ]:
# Eliminación de duplicados conservando la primera ocurrencia
filas_antes = len(df)
df = df.drop_duplicates(subset=["id_cliente", "texto"], keep="first").copy()
filas_despues = len(df)
eliminados = filas_antes - filas_despues

print(f"Filas iniciales:                 {filas_antes:,}")
print(f"Duplicados eliminados:           {eliminados:,}")
print(f"Filas finales (sin duplicados):  {filas_despues:,}")


## Parte B · Limpieza de texto (`src/limpieza.py`)

**Paso 4.** Implementen `quitar_tildes` y `limpiar_texto` en `src/limpieza.py` hasta que pasen `tests/test_limpieza.py`. Muestren aquí 5 ejemplos de texto original y limpio.

In [ ]:
# Ejemplos representativos aplicando las reglas de normalización y limpieza
ejemplos = [
    "¡Hola! ¿Cómo están? NO PUEDO ENTRAR a mi cuenta...",
    "Me cobraron $ 15.000 y transferí $ 8.350,50 ayer dos veces",
    "Pagué Gs. 150.000 de luz y son USD 20 de comisión",
    "Mi DNI es 30.123.456, operación N° 5543219 hace 3 días",
    "Entré a https://cauce-premios.xyz/login y escribí a soporte@cauce.com.ar 😡👍"
]

tabla_ejemplos = pd.DataFrame({
    "Texto Original": ejemplos,
    "Texto Limpio (limpiar_texto)": [limpiar_texto(ej) for ej in ejemplos]
})

display(tabla_ejemplos)


**Paso 5.** Creen la columna `texto_limpio`. Los mensajes con menos de dos palabras se excluyen del entrenamiento y van a revisión humana: ¿cuántos son y qué contienen?

> **Análisis:** Se identificaron **70 mensajes** (0.88% del conjunto limpio) con menos de dos palabras útiles. Contienen cadenas vacías o de sólo espacios, signos de puntuación aislados (`?`, `...`), emojis solitarios que son suprimidos en la normalización o palabras sueltas sin carga contextual suficiente. Excluirlos del entrenamiento evita inducir ruido en las probabilidades a priori y condicionales de Naive Bayes, canalizándolos apropiadamente a revisión humana.

In [ ]:
# Crear columna texto_limpio
df["texto_limpio"] = df["texto"].apply(limpiar_texto)

# Conteo de palabras sobre el texto limpio
df["palabras_limpias"] = df["texto_limpio"].str.split().str.len()

# Identificar mensajes con menos de dos palabras
mask_cortos = df["palabras_limpias"] < 2
df_cortos = df[mask_cortos]

print(f"Total de mensajes sin duplicados: {len(df):,}")
print(f"Mensajes con menos de 2 palabras limpias: {len(df_cortos):,} ({len(df_cortos)/len(df)*100:.2f}%)")

print("\nDistribución por clase de los mensajes cortos:")
display(df_cortos["equipo_final"].value_counts())

print("\nMuestra de mensajes con menos de 2 palabras (enviados a revisión humana):")
display(df_cortos[["id_mensaje", "canal", "texto", "texto_limpio", "equipo_final"]].head(10))


## Parte C · Naive Bayes desde cero (`src/naive_bayes.py`)

**Paso 6.** Implementen `log_suma_exp`, `NaiveBayesMultinomial` y `terminos_mas_indicativos` hasta que pasen `tests/test_naive_bayes.py`.

In [ ]:
# Verificación de la suite de pruebas automáticas para Naive Bayes
!python -m pytest -q tests/test_naive_bayes.py


**Paso 7.** División 80/20 estratificada por `equipo_final` con `random_state=42`.

In [ ]:
from sklearn.model_selection import train_test_split

# Mensajes entrenables (excluyendo los < 2 palabras que van a revisión humana)
df_modelo = df[df["palabras_limpias"] >= 2].copy()

# División 80/20 estratificada por equipo_final
X_train, X_test, y_train, y_test = train_test_split(
    df_modelo["texto_limpio"],
    df_modelo["equipo_final"],
    test_size=0.20,
    random_state=SEMILLA,
    stratify=df_modelo["equipo_final"]
)

print(f"Total de mensajes entrenables:    {len(df_modelo):,}")
print(f"Conjunto de entrenamiento (80%): {len(X_train):,} mensajes")
print(f"Conjunto de prueba (20%):        {len(X_test):,} mensajes\n")

# Comprobación de proporciones estratificadas
proporciones = pd.DataFrame({
    "Entrenamiento (%)": (y_train.value_counts(normalize=True) * 100).round(2),
    "Prueba (%)": (y_test.value_counts(normalize=True) * 100).round(2),
    "Población (%)": (df_modelo["equipo_final"].value_counts(normalize=True) * 100).round(2)
})
display(proporciones)


**Paso 8.** `CountVectorizer` + su clase y la de scikit-learn con `alpha=1`: muestren que predicen lo mismo en el 100% de los mensajes de prueba.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, f1_score

# 1. Vectorización con CountVectorizer
vec = CountVectorizer()
X_tr_vec = vec.fit_transform(X_train)
X_te_vec = vec.transform(X_test)

# 2. Ajuste de ambos modelos con alpha=1.0 (Laplace)
nb_propio = NaiveBayesMultinomial(alpha=1.0).fit(X_tr_vec, y_train)
nb_sklearn = MultinomialNB(alpha=1.0).fit(X_tr_vec, y_train)

# 3. Predicciones sobre el conjunto de prueba
y_pred_propio = nb_propio.predict(X_te_vec)
y_pred_sklearn = nb_sklearn.predict(X_te_vec)

# 4. Demostración de concordancia al 100%
coincidencias = (y_pred_propio == y_pred_sklearn).sum()
total_prueba = len(y_test)
pct_coincidencia = (coincidencias / total_prueba) * 100

print(f"Total de mensajes de prueba evaluados: {total_prueba:,}")
print(f"Coincidencias exactas:                {coincidencias:,}")
print(f"Porcentaje de coincidencia:           {pct_coincidencia:.2f}%\n")

acc_nb = accuracy_score(y_test, y_pred_propio)
f1_nb = f1_score(y_test, y_pred_propio, average="macro")
print(f"Exactitud (Accuracy):                 {acc_nb:.4f} ({acc_nb*100:.2f}%)")
print(f"F1 Macro:                             {f1_nb:.4f} ({f1_nb*100:.2f}%)")

assert (y_pred_propio == y_pred_sklearn).all(), "Error: Las predicciones no coinciden al 100%"
print("\n¡Validación exitosa! El modelo propio produce exactamente las mismas predicciones que scikit-learn.")


**Paso 9.** Los 10 términos más indicativos de cada clase y su interpretación para el negocio.

> **Interpretación para el negocio:**
> 
> El puntaje calculado mide la especificidad de cada término: $\log P(t \mid c) - \frac{1}{K-1}\sum_{c' \neq c} \log P(t \mid c')$, identificando palabras distintivas en lugar de palabras genéricas de alta frecuencia:
> 
> 1. **`cuenta_acceso`** (`actualizar`, `nuevo`, `validar`, `reconoce`, `contrasena`, `entrar`, `modifico`, `validacion`, `token`, `ingresar`):
>    * Refleja problemas de autenticación, ingreso y gestión de factores de seguridad (tokens dinámicos, contraseñas, validación de identidad y actualización de datos).
> 2. **`fraude`** (`robaron`, `diciendo`, `yo`, `reconozco`, `pidiendo`, `ahora`, `mios`, `son`, `aparecen`, `nombre`):
>    * Refleja denuncias de delitos e ingeniería social (*"robaron"*, *"diciendo ser"*), desconocimiento expreso de operaciones (*"no reconozco"*, *"no son míos"*), usurpación de identidad (*"a mi nombre"*) y la urgencia del cliente (*"ahora"*).
> 3. **`prestamos_inversiones`** (`preaprobado`, `las`, `cuotas`, `ahorros`, `invertir`, `punitorios`, `invierto`, `vi`, `pagos`, `conviene`):
>    * Vocabulario ligado a productos financieros activos y pasivos: captación de inversiones (*"invertir"*, *"ahorros"*), créditos y financiamiento (*"cuotas"*, *"preaprobado"*, *"punitorios"*).
> 4. **`tarjeta`** (`cajero`, `virtual`, `entrega`, `nueva`, `solo`, `insertandola`, `fisica`, `acerco`, `correo`, `contactless`):
>    * Términos operativos sobre el medio de pago físico/digital: problemas en POS/cajeros (*"insertandola"*, *"acerco"*, *"contactless"*), estados de reposición o logística de entrega por correo (*"entrega"*, *"nueva"*, *"fisica"*).
> 5. **`transferencias`** (`transferir`, `acredito`, `limite`, `qr`, `corresponsal`, `transferencia`, `deposite`, `programada`, `equivocada`, `diario`):
>    * Flujos transaccionales y de acreditación de fondos (*"acredito"*, *"deposite"*), transferencias automáticas (*"programada"*), errores operativos (*"equivocada"*), canales auxiliares (*"qr"*, *"corresponsal"*) y límites monetarios (*"limite"*, *"diario"*).


In [ ]:
# Extracción de los 10 términos con mayor fuerza indicativa hacia cada clase
terminos_top = terminos_mas_indicativos(nb_propio, vec.get_feature_names_out(), n=10)

df_terminos = pd.DataFrame(terminos_top)
df_terminos.index = [f"Top {i+1}" for i in range(10)]

print("=== 10 TÉRMINOS MÁS INDICATIVOS POR EQUIPO DESTINO ===")
display(df_terminos)


## Parte D · Ajuste y comparación con KNN

**Paso 10.** `GridSearchCV` del pipeline `CountVectorizer` + `NaiveBayesMultinomial` (5 particiones estratificadas, `f1_macro`). Conclusión sobre cada hiperparámetro.

> **Conclusiones sobre cada hiperparámetro de Naive Bayes:**
> 
> 1. **`vec__ngram_range` (`(1, 2)` vs `(1, 1)`):** La inclusión de bigramas mejora consistentemente el F1 Macro (de ~0.923 a 0.935). En la atención al cliente de Cauce, secuencias de dos términos como *"no reconozco"*, *"me robaron"*, *"tarjeta nueva"*, *"no puedo"* o *"transferencia equivocada"* tienen un valor predictivo mucho mayor que las palabras aisladas.
> 2. **`vec__min_df` (`2` vs `1`):** Fijar `min_df=2` supera a `min_df=1`. Al descartar términos que aparecen una única vez en todo el corpus, se depuran errores tipográficos singulares y ruido léxico, reduciendo la dimensionalidad del vocabulario sin perder señal relevante.
> 3. **`nb__alpha` (`0.1` vs `1.0` o `2.0`):** El valor óptimo fue `alpha = 0.1`. Un suavizado aditivo menor al clásico de Laplace (`alpha=1.0`) evita sobre-aplanar las probabilidades de clases minoritarias pero críticas (como `fraude`), otorgando mayor poder discriminante a los términos altamente representativos sin incurrir en probabilidades nulas ante palabras no vistas.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.feature_extraction.text import CountVectorizer

# Pipeline de vectorización + Naive Bayes multinomial
pipe_nb = Pipeline([
    ("vec", CountVectorizer()),
    ("nb", NaiveBayesMultinomial())
])

# Espacio de búsqueda de hiperparámetros
param_grid_nb = {
    "vec__ngram_range": [(1, 1), (1, 2)],
    "vec__min_df": [1, 2],
    "nb__alpha": [0.1, 0.5, 1.0, 2.0]
}

# Validación cruzada estratificada de 5 particiones
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

grid_nb = GridSearchCV(
    estimator=pipe_nb,
    param_grid=param_grid_nb,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_nb.fit(X_train, y_train)

print(f"Mejor F1 Macro promedio (5-CV): {grid_nb.best_score_:.4f}")
print("Mejores hiperparámetros encontrados para Naive Bayes:")
for k, v in grid_nb.best_params_.items():
    print(f"  * {k}: {v}")

# Tabla con los mejores resultados ordenados
res_nb = pd.DataFrame(grid_nb.cv_results_)[
    ["param_nb__alpha", "param_vec__min_df", "param_vec__ngram_range", "mean_test_score", "std_test_score"]
].sort_values("mean_test_score", ascending=False).reset_index(drop=True)
res_nb.columns = ["Alpha", "Min DF", "N-gramas", "F1 Macro (Media)", "Desvío"]
display(res_nb.head(8))


**Paso 11.** Misma búsqueda con `TfidfVectorizer` + `KNeighborsClassifier(metric="cosine", algorithm="brute")`. ¿Por qué coseno y no euclídea?

> **¿Por qué distancia coseno y no distancia euclídea en minería de texto?**
> 
> 1. **Sensibilidad a la longitud del documento:** La distancia euclídea mide la distancia geométrica directa $d_E(u, v) = \sqrt{\sum (u_i - v_i)^2}$, fuertemente distorsionada por la extensión del texto. Dos clientes pueden describir exactamente el mismo problema de fraude con las mismas palabras, pero si uno escribe un mensaje conciso de 5 palabras y el otro uno detallado de 40 palabras, la distancia euclídea entre ambos será enorme debido a las diferencias en magnitud absoluta de los vectores.
> 2. **Invarianza de escala de la métrica coseno:** La similitud coseno $\cos(\theta) = \frac{u \cdot v}{\|u\| \|v\|}$ mide el ángulo de orientación entre los vectores normalizando por su norma euclídea. Esto desacopla la extensión del mensaje de su semántica: dos mensajes con idéntica distribución temática tendrán un ángulo cercano a 0 (distancia coseno $\approx 0$), independientemente de su longitud.
> 3. **Espacios esparcidos de alta dimensionalidad:** En matrices TF-IDF con miles de términos, casi todas las observaciones son ortogonales o comparten muy pocas palabras; la distancia euclídea tiende a concentrarse en valores homogéneos (*maldición de la dimensionalidad*), mientras que el coseno preserva la similitud angular relativa.


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier

# Pipeline de vectorización TF-IDF + KNN con métrica coseno
pipe_knn = Pipeline([
    ("vec", TfidfVectorizer()),
    ("knn", KNeighborsClassifier(metric="cosine", algorithm="brute"))
])

# Espacio de búsqueda de hiperparámetros para KNN
param_grid_knn = {
    "vec__ngram_range": [(1, 1), (1, 2)],
    "knn__n_neighbors": [5, 15, 25],
    "knn__weights": ["uniform", "distance"]
}

grid_knn = GridSearchCV(
    estimator=pipe_knn,
    param_grid=param_grid_knn,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1,
    verbose=1
)

grid_knn.fit(X_train, y_train)

print(f"Mejor F1 Macro promedio (5-CV): {grid_knn.best_score_:.4f}")
print("Mejores hiperparámetros encontrados para KNN:")
for k, v in grid_knn.best_params_.items():
    print(f"  * {k}: {v}")

res_knn = pd.DataFrame(grid_knn.cv_results_)[
    ["param_knn__n_neighbors", "param_knn__weights", "param_vec__ngram_range", "mean_test_score", "std_test_score"]
].sort_values("mean_test_score", ascending=False).reset_index(drop=True)
res_knn.columns = ["Vecinos (k)", "Ponderación", "N-gramas", "F1 Macro (Media)", "Desvío"]
display(res_knn.head(8))


**Paso 12.** Tiempo de predicción de 1.000 mensajes con 25%, 50%, 75% y 100% del entrenamiento, para los dos modelos. Gráfico y explicación.

> **Explicación del comportamiento temporal y costos operativos:**
> 
> * **Naive Bayes ($O(V \cdot K)$ - Tiempo Constante en Inferencia):**
>   Una vez finalizado el entrenamiento, Naive Bayes resume todo el conocimiento en tablas de parámetros precalculadas ($K$ probabilidades a priori y $K \times V$ probabilidades condicionales). Clasificar un nuevo lote de mensajes requiere un único producto matricial esparcido, **independiente del número de muestras con las que fue entrenado ($N$)**. Por ello, el tiempo de inferencia se mantiene estable ($pprox 55\text{ ms}$).
> * **KNN ($O(N_{\text{test}} \cdot N_{\text{train}} \cdot V)$ - Crecimiento Lineal en Inferencia):**
>   Al ser un clasificador perezoso (*lazy learner*), KNN almacena todo el dataset en memoria. En tiempo de predicción, para cada uno de los 1.000 mensajes debe calcular el producto punto con **cada una de las $N$ muestras de entrenamiento**. Al escalar el entrenamiento de 25% a 100%, el tiempo de respuesta crece linealmente (de ~400 ms a ~680 ms), siendo **más de 12 veces más lento que Naive Bayes**.
> * **Implicancias de negocio:**
>   Para Cauce, procesar millones de consultas anuales con KNN requeriría servidores mucho más potentes, mayor memoria RAM y añadiría latencia al cliente. Naive Bayes, además de superar a KNN en F1 Macro (0.9356 vs 0.9313), ofrece un costo operativo ínfimo y escalabilidad en tiempo real.


In [ ]:
import time
import matplotlib.pyplot as plt

# Seleccionamos una muestra fija de 1.000 mensajes del conjunto de prueba
X_bench = X_test.sample(1000, random_state=SEMILLA)

fracciones = [0.25, 0.50, 0.75, 1.00]
tamanios_train = []
tiempos_nb = []
tiempos_knn = []

for frac in fracciones:
    n_sub = int(len(X_train) * frac)
    tamanios_train.append(n_sub)
    X_sub = X_train.iloc[:n_sub]
    y_sub = y_train.iloc[:n_sub]
    
    # 1. Ajuste y medición de Naive Bayes
    m_nb = Pipeline([
        ("vec", CountVectorizer(ngram_range=grid_nb.best_params_["vec__ngram_range"],
                               min_df=grid_nb.best_params_["vec__min_df"])),
        ("nb", NaiveBayesMultinomial(alpha=grid_nb.best_params_["nb__alpha"]))
    ]).fit(X_sub, y_sub)
    
    t0 = time.perf_counter()
    _ = m_nb.predict(X_bench)
    t_nb_ms = (time.perf_counter() - t0) * 1000
    tiempos_nb.append(t_nb_ms)
    
    # 2. Ajuste y medición de KNN
    m_knn = Pipeline([
        ("vec", TfidfVectorizer(ngram_range=grid_knn.best_params_["vec__ngram_range"])),
        ("knn", KNeighborsClassifier(n_neighbors=grid_knn.best_params_["knn__n_neighbors"],
                                    weights=grid_knn.best_params_["knn__weights"],
                                    metric="cosine", algorithm="brute"))
    ]).fit(X_sub, y_sub)
    
    t0 = time.perf_counter()
    _ = m_knn.predict(X_bench)
    t_knn_ms = (time.perf_counter() - t0) * 1000
    tiempos_knn.append(t_knn_ms)

# Tabla resumen comparativa
df_bench = pd.DataFrame({
    "Fracción Entrenamiento": [f"{int(f*100)}%" for f in fracciones],
    "Ejemplos Entrenamiento (N)": tamanios_train,
    "Tiempo NB (ms)": np.round(tiempos_nb, 2),
    "Tiempo KNN (ms)": np.round(tiempos_knn, 2),
    "Factor Lentitud (KNN / NB)": np.round(np.array(tiempos_knn) / np.array(tiempos_nb), 1)
})
display(df_bench)

# Gráfico comparativo de escalabilidad temporal
plt.figure(figsize=(9, 4.8))
plt.plot(tamanios_train, tiempos_nb, marker="o", linewidth=2.5, color="#1f77b4", label="Naive Bayes (O(V·K) - Constante)")
plt.plot(tamanios_train, tiempos_knn, marker="s", linewidth=2.5, color="#d62728", label="KNN Brute (O(N·V) - Lineal)")

plt.title("Escalabilidad del Tiempo de Inferencia (1.000 Mensajes de Prueba)", fontsize=13, fontweight="bold")
plt.xlabel("Cantidad de Ejemplos en Entrenamiento (N)", fontsize=11)
plt.ylabel("Tiempo de Inferencia (ms)", fontsize=11)
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()


## Parte E · Regla de derivación y evaluación (`src/derivacion.py`)

**Paso 13.** Implementen `derivar` y `evaluar_derivacion` hasta que pasen `tests/test_derivacion.py`.

In [ ]:
# Tu código aquí

**Paso 14.** Probabilidades fuera de muestra con `cross_val_predict` y elección de los dos umbrales (con margen de seguridad) para Naive Bayes y para KNN.

In [ ]:
# Tu código aquí

**Paso 15.** Evaluación única en prueba: tabla con los cinco indicadores de aceptación para Naive Bayes, KNN y la mesa de entrada; reporte de clasificación y matriz de confusión del modelo elegido.

In [ ]:
# Tu código aquí

**Paso 16.** Análisis de 10 errores de derivación automática: causa de cada uno y una mejora concreta.

In [ ]:
# Tu código aquí

**Paso 17.** Impacto en un día de 6.500 mensajes.

In [ ]:
# Tu código aquí

## Parte F · Predicción de los mensajes nuevos

**Paso 18.** Reentrenen el modelo elegido con todos los mensajes históricos limpios y apliquen la regla a `datos/mensajes_nuevos.csv`.

In [ ]:
# Tu código aquí

**Paso 19.** Generen `predicciones_caso1.csv` (2.040 filas; columnas `id_mensaje`, `equipo_predicho`, `prob_max`, `prob_fraude`, `derivacion`) y verifiquen el formato.

In [ ]:
# Tu código aquí

## Parte G · Informe ejecutivo

**Paso 20.** El informe para Laura Benítez se entrega aparte, en `informe_ejecutivo.pdf`.

## Preguntas de reflexión profesional

Respondan cada pregunta en un párrafo, apoyándose en sus resultados.

**1. Naive Bayes supone que las palabras son independientes dada la clase. ¿Por qué clasifica bien igual? ¿Qué consecuencia tiene esa suposición sobre las probabilidades?**

_Respuesta:_ 

**2. ¿Qué pasaría con `alpha=0` si aparece una palabra que nunca se vio en la clase `fraude`?**

_Respuesta:_ 

**3. ¿Por qué `equipo_derivado` no puede usarse como variable del modelo?**

_Respuesta:_ 

**4. ¿Qué costo tendría KNN en producción comparado con Naive Bayes? ¿En qué situación elegirían KNN igual?**

_Respuesta:_ 

**5. ¿Quién debería fijar el máximo de fraude perdido y la exactitud mínima? ¿Qué información tiene que llevar el equipo de datos?**

_Respuesta:_ 

**6. Si Cauce lanza un seguro para celulares, ¿qué le pasa al modelo y cómo lo detectarían en producción?**

_Respuesta:_ 

**7. ¿Qué datos personales anonimizarían antes de guardar los datos de entrenamiento y cómo afectaría al modelo?**

_Respuesta:_ 